# SO-101 Joint State Publisher

Publishes a `sensor_msgs/JointState` message for the SO-101 robot arm
based on the joint definitions in `bambot/website/public/URDFs/so101.urdf`.

| Joint | Lower (rad) | Upper (rad) | Neutral (mid) |
|-------------|-------------|-------------|---------------|
| Rotation | 1.22014 | 5.05986 | 3.14000 |
| Pitch | 1.39467 | 4.88692 | 3.14080 |
| Elbow | 1.39626 | 4.71239 | 3.05433 |
| Wrist_Pitch | 1.48353 | 4.79965 | 3.14159 |
| Wrist_Roll | 0.39774 | 5.98280 | 3.19027 |
| Jaw | 3.14000 | 4.88692 | 4.01346 |

In [1]:
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import JointState
from builtin_interfaces.msg import Time

# Joint names and limits from so101.urdf (revolute joints only)
JOINTS = [
    # name          lower     upper
    ("Rotation",    1.22014,  5.05986),
    ("Pitch",       1.39467,  4.88692),
    ("Elbow",       1.39626,  4.71239),
    ("Wrist_Pitch", 1.48353,  4.79965),
    ("Wrist_Roll",  0.39774,  5.98280),
    ("Jaw",         3.14000,  4.88692),
]

JOINT_NAMES = [j[0] for j in JOINTS]
NEUTRAL_POSITIONS = [(j[1] + j[2]) / 2.0 for j in JOINTS]  # midpoint of each range

print("Joint names:    ", JOINT_NAMES)
print("Neutral positions (rad):", [round(p, 5) for p in NEUTRAL_POSITIONS])

Joint names:     ['Rotation', 'Pitch', 'Elbow', 'Wrist_Pitch', 'Wrist_Roll', 'Jaw']
Neutral positions (rad): [3.14, 3.14079, 3.05432, 3.14159, 3.19027, 4.01346]


In [2]:
class JointStatePublisher(Node):
    def __init__(self, positions: list[float]):
        super().__init__("so101_joint_state_publisher")
        self.pub = self.create_publisher(JointState, "/joint_states", 10)
        self.positions = positions
        # Publish once immediately, then on a 10 Hz timer
        self._publish()
        self.timer = self.create_timer(0.1, self._publish)

    def _publish(self):
        msg = JointState()
        msg.header.stamp = self.get_clock().now().to_msg()
        msg.name = JOINT_NAMES
        msg.position = self.positions
        msg.velocity = [0.0] * len(JOINT_NAMES)
        msg.effort = [0.0] * len(JOINT_NAMES)
        self.pub.publish(msg)
        self.get_logger().info(
            "Published JointState: " + str([round(p, 4) for p in self.positions])
        )


if not rclpy.ok():
    rclpy.init(args=None)

node = JointStatePublisher(NEUTRAL_POSITIONS)

try:
    # Spin for a few cycles then stop — change to rclpy.spin(node) for continuous publishing
    for _ in range(5):
        rclpy.spin_once(node, timeout_sec=0.2)
except KeyboardInterrupt:
    pass
finally:
    node.destroy_node()
    rclpy.shutdown()

[INFO] [1774019157.262981512] [so101_joint_state_publisher]: Published JointState: [3.14, 3.1408, 3.0543, 3.1416, 3.1903, 4.0135]
[INFO] [1774019157.364741499] [so101_joint_state_publisher]: Published JointState: [3.14, 3.1408, 3.0543, 3.1416, 3.1903, 4.0135]
[INFO] [1774019157.464939293] [so101_joint_state_publisher]: Published JointState: [3.14, 3.1408, 3.0543, 3.1416, 3.1903, 4.0135]
[INFO] [1774019157.564728937] [so101_joint_state_publisher]: Published JointState: [3.14, 3.1408, 3.0543, 3.1416, 3.1903, 4.0135]
[INFO] [1774019157.664800279] [so101_joint_state_publisher]: Published JointState: [3.14, 3.1408, 3.0543, 3.1416, 3.1903, 4.0135]
[INFO] [1774019157.764799374] [so101_joint_state_publisher]: Published JointState: [3.14, 3.1408, 3.0543, 3.1416, 3.1903, 4.0135]


## Publish a custom pose

Edit the `custom_positions` dict and run the cell to publish specific joint angles.
All values are in **radians** and must stay within the limits shown in the table above.

In [3]:
custom_positions = {
    "Rotation":    3.14159,
    "Pitch":       3.14159,
    "Elbow":       3.05433,
    "Wrist_Pitch": 3.14159,
    "Wrist_Roll":  3.19027,
    "Jaw":         4.01346,
}

# Validate against limits
for name, lower, upper in JOINTS:
    val = custom_positions[name]
    assert lower <= val <= upper, (
        f"{name} = {val:.5f} is outside [{lower}, {upper}]"
    )

if not rclpy.ok():
    rclpy.init(args=None)

node = JointStatePublisher([custom_positions[n] for n in JOINT_NAMES])

try:
    for _ in range(5):
        rclpy.spin_once(node, timeout_sec=0.2)
except KeyboardInterrupt:
    pass
finally:
    node.destroy_node()
    rclpy.shutdown()

[INFO] [1774019157.779096903] [so101_joint_state_publisher]: Published JointState: [3.1416, 3.1416, 3.0543, 3.1416, 3.1903, 4.0135]
[INFO] [1774019157.880755940] [so101_joint_state_publisher]: Published JointState: [3.1416, 3.1416, 3.0543, 3.1416, 3.1903, 4.0135]
[INFO] [1774019157.980885474] [so101_joint_state_publisher]: Published JointState: [3.1416, 3.1416, 3.0543, 3.1416, 3.1903, 4.0135]
[INFO] [1774019158.080784595] [so101_joint_state_publisher]: Published JointState: [3.1416, 3.1416, 3.0543, 3.1416, 3.1903, 4.0135]
[INFO] [1774019158.180823330] [so101_joint_state_publisher]: Published JointState: [3.1416, 3.1416, 3.0543, 3.1416, 3.1903, 4.0135]
[INFO] [1774019158.280852374] [so101_joint_state_publisher]: Published JointState: [3.1416, 3.1416, 3.0543, 3.1416, 3.1903, 4.0135]


In [4]:
import threading
import ipywidgets as widgets
from IPython.display import display

# --- ROS node (persistent across slider interactions) ---
if not rclpy.ok():
    rclpy.init(args=None)

if "slider_node" not in globals() or not rclpy.ok():
    slider_node = rclpy.create_node("so101_slider_publisher")
    slider_pub = slider_node.create_publisher(JointState, "/joint_states", 10)

def publish_joint_state(positions: dict[str, float]):
    msg = JointState()
    msg.header.stamp = slider_node.get_clock().now().to_msg()
    msg.name = JOINT_NAMES
    msg.position = [positions[n] for n in JOINT_NAMES]
    msg.velocity = [0.0] * len(JOINT_NAMES)
    msg.effort = [0.0] * len(JOINT_NAMES)
    slider_pub.publish(msg)

# --- Build one slider per joint ---
sliders = {
    name: widgets.FloatSlider(
        value=(lower + upper) / 2.0,
        min=lower,
        max=upper,
        step=0.001,
        description=name,
        continuous_update=True,
        style={"description_width": "100px"},
        layout=widgets.Layout(width="500px"),
    )
    for name, lower, upper in JOINTS
}

def on_change(change):
    publish_joint_state({n: s.value for n, s in sliders.items()})

for s in sliders.values():
    s.observe(on_change, names="value")

reset_btn = widgets.Button(description="Reset to neutral", button_style="info")

def on_reset(_):
    for (name, lower, upper), s in zip(JOINTS, sliders.values()):
        s.value = (lower + upper) / 2.0

reset_btn.on_click(on_reset)

display(widgets.VBox(list(sliders.values()) + [reset_btn]))